# N-ATLaS server for Safroi (free Colab T4)

Serves `NCAIR1/N-ATLaS` (4-bit) as an OpenAI-style `/v1/chat/completions` API behind a free Cloudflare tunnel.

**Before you run:**
1. Runtime → Change runtime type → **T4 GPU**.
2. Left sidebar → 🔑 **Secrets** → add two secrets and turn **Notebook access ON** for both:
   - `HF_TOKEN`: your Hugging Face token (the account that was granted access to N-ATLaS)
   - `NATLAS_API_KEY`: any long random string you invent. Safroi will send it as the Bearer token.
3. Run the cells top to bottom. The last cell prints your public URL.

Free Colab disconnects when idle and after ~12 h, and the URL changes each run. Use this for demos/testing.

*N-ATLaS is an initiative of the Federal Ministry of Communications, Innovation and Digital Economy, and powered by Awarri Technologies.*

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes fastapi uvicorn

In [ ]:
import torch
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

HF_TOKEN = userdata.get('HF_TOKEN')
API_KEY = userdata.get('NATLAS_API_KEY')
assert API_KEY and len(API_KEY) >= 16, 'Set a NATLAS_API_KEY secret (16+ chars)'
login(HF_TOKEN)

MODEL = 'NCAIR1/N-ATLaS'
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True),
    device_map='auto',
)
model.eval()
print('Loaded. GPU memory used: %.1f GB' % (torch.cuda.memory_allocated() / 1e9))

In [ ]:
import threading, time, uuid
from datetime import datetime
from typing import List, Optional
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel

MAX_INPUT_TOKENS = 7000        # model context is ~8k; leave room for the answer
lock = threading.Lock()        # one generation at a time on a single T4
app = FastAPI()

class Msg(BaseModel):
    role: str
    content: str

class ChatReq(BaseModel):
    model: Optional[str] = None
    messages: List[Msg]
    max_tokens: Optional[int] = 512
    temperature: Optional[float] = 0.1

def check(auth):
    if auth != f'Bearer {API_KEY}':
        raise HTTPException(401, 'Unauthorized')

@app.get('/health')
def health():
    return {'status': 'ok', 'model': MODEL}

@app.post('/v1/chat/completions')
def chat(req: ChatReq, authorization: Optional[str] = Header(None)):
    check(authorization)
    text = tok.apply_chat_template(
        [m.dict() for m in req.messages], add_generation_prompt=True, tokenize=False,
        date_string=datetime.now().strftime('%d %b %Y'))
    inputs = tok(text, return_tensors='pt', add_special_tokens=False).to(model.device)
    n_in = inputs['input_ids'].shape[1]
    if n_in > MAX_INPUT_TOKENS:
        raise HTTPException(400, f'Input too long ({n_in} tokens, max {MAX_INPUT_TOKENS}).')
    temp = req.temperature or 0
    gen = dict(max_new_tokens=min(req.max_tokens or 512, 1024), repetition_penalty=1.12, use_cache=True)
    gen.update(dict(do_sample=True, temperature=temp) if temp > 0.01 else dict(do_sample=False))
    with lock, torch.inference_mode():
        out = model.generate(**inputs, **gen)
    new = out[0][n_in:]
    answer = tok.decode(new, skip_special_tokens=True).strip()
    return {
        'id': 'chatcmpl-' + uuid.uuid4().hex, 'object': 'chat.completion',
        'created': int(time.time()), 'model': MODEL,
        'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': answer}, 'finish_reason': 'stop'}],
        'usage': {'prompt_tokens': n_in, 'completion_tokens': len(new), 'total_tokens': n_in + len(new)},
    }

In [ ]:
import subprocess, threading, re, time, uvicorn

# Run the API on localhost:8000 in a background thread
config = uvicorn.Config(app, host='127.0.0.1', port=8000, log_level='warning')
server = uvicorn.Server(config)
threading.Thread(target=server.run, daemon=True).start()
time.sleep(3)

# Free Cloudflare quick tunnel (no account needed)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
proc = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in proc.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0); break
print('\nNATLAS_BASE_URL =', url + '/v1')
print('NATLAS_API_KEY  = (the secret you created)')

In [ ]:
# Smoke test through the public URL (Hausa: "what is meant by gwagwarmaya")
import requests
r = requests.post(url + '/v1/chat/completions',
    headers={'Authorization': f'Bearer {API_KEY}'},
    json={'messages': [{'role': 'user', 'content': 'Translate to Hausa: The company can close your account at any time without warning.'}],
          'max_tokens': 120}, timeout=300)
print(r.status_code, r.json()['choices'][0]['message']['content'])

Keep this tab open. If it disconnects, rerun all cells and update `NATLAS_BASE_URL` in Safroi's environment.